In [1]:
with open('text.txt', 'r', encoding='utf-8') as file:
    text = file.read()


In [2]:
text[:500]

'Əziz Samir,\n\nNaxçıvan-dan sənə avqust ayında yazmağa qərar verdim.\nBuradakı həyat hər gün daha da dəyişir.\nMən çoban kimi işləməyə davam edirəm, amma ürəyim həmişə sənin yanındadır.\n\nKeçən ay baş verənlər məni çox düşündürdü.\nHəyəcan hissiyyatım var ürəyimdə, onu tam ifadə etmək çətindir.\nUmarım sən yaxşısan, ailən sağlamdır.\n\nBir gün yenidən görüşərik, əminəm.\nO günə qədər sağ ol,\n\nZəhra\n\nBir varmış, bir yoxmuş. Quba yaxınlığında böyük bir bulaq kənarında kiçik bir kənd varmış.\nO kənddə Tural a'

In [3]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("".join(chars))
print(len(chars))


 "(),-.:?ABCDEFGHIKLMNOPQRSTUVXYZabcdefghijklmnopqrstuvxyzÇÜçöüğİıŞşƏə—
72


In [4]:
stoi = {ch:i for i,ch in enumerate(chars)} #str to i
itos = {i:ch for i,ch in enumerate(chars)} #i to str
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join([itos[i] for i in l])

print(encode('salam'))
print(decode(encode('salam')))

[52, 34, 45, 34, 46]
salam


In [5]:
import torch 
data = torch.tensor(encode(text),dtype=torch.long)
print(len(data))
print(data[:50])

                    

889939
tensor([69, 58, 42, 58,  1, 27, 34, 46, 42, 51,  5,  0,  0, 22, 34, 56, 61, 66,
        55, 34, 47,  6, 37, 34, 47,  1, 52, 70, 47, 70,  1, 34, 55, 50, 54, 52,
        53,  1, 34, 57, 66, 47, 37, 34,  1, 57, 34, 58, 46, 34])


In [6]:
n = int(0.9*len(data))
train = data[:n]
validation = data[n:]

In [7]:
block_size = 8
x = train[:block_size]
y = train[1:block_size+1]

for i in range(block_size):
    context = x[:i+1]
    target = y[i]
    print(f"{context},    {target}")  
    
    
# BUNU ETMEYIMIZIN DIGER SEBEBINE TRANSFORMERSE BUTUN CUMLENI TOKEN TOKEN GOSTERMEK ISTEYIRIK
# EN KICIK DEN BASLASIN EN BOYUK SAYDA TOKENI TAMAMLASIN

tensor([69]),    58
tensor([69, 58]),    42
tensor([69, 58, 42]),    58
tensor([69, 58, 42, 58]),    1
tensor([69, 58, 42, 58,  1]),    27
tensor([69, 58, 42, 58,  1, 27]),    34
tensor([69, 58, 42, 58,  1, 27, 34]),    46
tensor([69, 58, 42, 58,  1, 27, 34, 46]),    42


In [ ]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
    data = train if split == 'train' else validation
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x,y

xb,yb = get_batch('train')
print(xb,yb)


tensor([[45, 70, 51, 42,  8,  0,  0, 27],
        [38, 45,  1, 50, 34, 51, 37, 34],
        [ 1, 46, 42, 47,  1, 42, 45, 45],
        [34, 35, 70, 51,  1, 50, 70, 35]]) tensor([[70, 51, 42,  8,  0,  0, 27, 54],
        [45,  1, 50, 34, 51, 37, 34, 68],
        [46, 42, 47,  1, 42, 45, 45, 70],
        [35, 70, 51,  1, 50, 70, 35, 54]])


In [ ]:
import torch.nn as nn
from torch.nn import functional as F 
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self,vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)

    
    def forward(self,idx,targets=None):
        logits = self.token_embedding_table(idx)
        
        if targets == None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits,targets)

        return logits,loss
    
    def generate(self,idx,max_new_tokens):
        for _ in range(max_new_tokens):
            logits,loss = self(idx)
            logits = logits[:,-1,:]
            probs = F.softmax(logits,dim=-1)
            idx_next = torch.multinomial(probs,num_samples=1)
            idx = torch.cat((idx,idx_next),dim=1)
        return idx

    
m =  BigramLanguageModel(vocab_size)
out = m(xb,yb)
print(decode(m.generate(idx=torch.zeros((1,1),dtype=torch.long),max_new_tokens=100)[0].tolist()))


ö
öğ
-dX.Cc dUYvX DdTƏhəTfQmU(yDəHVVüÜÇÇəMOpumiOTNs:gk(TƏExukLüKGZXITSIGMlmHÇmÜkM—Xğ
M(I"
ləMsHVgVAQ


In [15]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [18]:
batch_size = 32
for steps in range(10000):
    xb,yb = get_batch('train')

    logits,loss = m(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.4659972190856934


In [19]:
print(decode(m.generate(idx=torch.zeros((1,1),dtype=torch.long),max_new_tokens=100)[0].tolist()))


SəngəncXordıbusadq adami stm Sab ndeyr. tət Çon t bölamiriş.
Nonlarəmi: hpəribdağQərdəratlətləkidədu
